In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("KICKSTARTER DURING-CAMPAIGN MONITOR - MODEL TRAINING")
print("=" * 80)

# Load data
print("\n[1/5] Loading data...")
df = pd.read_csv('df_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Display basic info
print(f"\nTarget variable (success) distribution:")
print(df['success'].value_counts() if 'success' in df.columns else "Success column not found")

# Define pre-campaign features (15 indicators) - NUMERIC ONLY
pre_campaign_features = [
    'goal',
    'log_goal',
    'usd_goal_real',
    'launch_month',
    'launch_quarter',
    'campaign_length',
    'is_optimal_duration',
    'is_weekend_launch',
    'is_us',
    'category_baseline_success',
    'country_baseline_success',
    'deadline_weekday'
]

# Check which features exist
available_features = [f for f in pre_campaign_features if f in df.columns]
print(f"\nAvailable features for training: {len(available_features)}/{len(pre_campaign_features)}")
print(f"Features: {available_features}")

# Prepare data
print("\n[2/5] Preparing data...")

# Use 'state' as target (convert to binary: successful=1, else=0)
if 'state' in df.columns:
    y = (df['state'] == 'successful').astype(int)
elif 'success' in df.columns:
    y = df['success'].copy()
else:
    print("ERROR: Neither 'state' nor 'success' column found in dataset!")
    print(f"Available columns: {df.columns.tolist()}")
    exit(1)

# Create missing features
print("  Creating missing features...")
df_temp = df.copy()

# Early performance features (Week 1-2)
if 'pledged' in df_temp.columns:
    df_temp['daily_pledges'] = df_temp['pledged'] / (df_temp['campaign_length'] + 1)
else:
    df_temp['daily_pledges'] = 0

if 'backers' in df_temp.columns:
    df_temp['daily_backers'] = df_temp['backers'] / (df_temp['campaign_length'] + 1)
    df_temp['backers_count'] = df_temp['backers']
else:
    df_temp['daily_backers'] = 0
    df_temp['backers_count'] = 0

# Conversion rate
if 'pledged' in df_temp.columns and 'goal' in df_temp.columns:
    df_temp['conversion_rate'] = df_temp['pledged'] / (df_temp['goal'] + 1)
else:
    df_temp['conversion_rate'] = 0

# Other features
df_temp['updates_count'] = 0  # Not in original data
df_temp['comments_count'] = 0  # Not in original data

# Encode categorical features
if 'category' in df_temp.columns:
    df_temp['category_encoded'] = pd.factorize(df_temp['category'])[0]
else:
    df_temp['category_encoded'] = 0

if 'country' in df_temp.columns:
    df_temp['country_encoded'] = pd.factorize(df_temp['country'])[0]
else:
    df_temp['country_encoded'] = 0

if 'currency' in df_temp.columns:
    df_temp['currency_encoded'] = pd.factorize(df_temp['currency'])[0]
else:
    df_temp['currency_encoded'] = 0

if 'campaign_length' in df_temp.columns:
    df_temp['days_to_deadline'] = df_temp['campaign_length']
else:
    df_temp['days_to_deadline'] = 0

df_temp['has_video'] = 0  # Not in original data

# Replace df for feature extraction
df = df_temp

X = df[available_features].copy()

# Handle missing values
print(f"Missing values before cleaning: {X.isnull().sum().sum()}")
X = X.fillna(X.median())
print(f"Missing values after cleaning: {X.isnull().sum().sum()}")

# Remove rows with NaN in target
valid_idx = ~y.isnull()
X = X[valid_idx]
y = y[valid_idx]

print(f"Final dataset shape: {X.shape}")
print(f"Success distribution:\n{y.value_counts()}")
print(f"Success rate: {y.mean():.2%}")

# Split data
print("\n[3/5] Splitting data (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Scale features
print("\n[4/5] Training models...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n  Training {name}...")
    
    # Use scaled data for Logistic Regression, original for tree models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'y_pred': y_pred
    }
    
    print(f"    ✓ Accuracy:  {accuracy:.2%}")
    print(f"    ✓ Precision: {precision:.2%}")
    print(f"    ✓ Recall:    {recall:.2%}")
    print(f"    ✓ F1-Score:  {f1:.2%}")

# Best model
print("\n[5/5] Results Summary")
print("=" * 80)
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
best_result = results[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy: {best_result['accuracy']:.2%}")
print(f"   Expected Performance: {best_result['accuracy']:.0%} (Target: 75-80%)")

print(f"\n📊 ALL MODELS COMPARISON:")
for name in sorted(results.keys(), key=lambda x: results[x]['accuracy'], reverse=True):
    res = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy:  {res['accuracy']:.2%}")
    print(f"    Precision: {res['precision']:.2%}")
    print(f"    Recall:    {res['recall']:.2%}")
    print(f"    F1-Score:  {res['f1']:.2%}")

# Detailed classification report for best model
print(f"\n📋 DETAILED REPORT ({best_model_name}):")
print("\n" + classification_report(y_test, best_result['y_pred'], 
                                   target_names=['Failed (0)', 'Succeeded (1)']))

# Confusion matrix
cm = confusion_matrix(y_test, best_result['y_pred'])
print(f"\nConfusion Matrix:")
print(f"  ┌─────────────┬──────────┬──────────┐")
print(f"  │             │ Pred: 0  │ Pred: 1  │")
print(f"  ├─────────────┼──────────┼──────────┤")
print(f"  │ Actual: 0   │ {cm[0,0]:8d} │ {cm[0,1]:8d} │")
print(f"  │ Actual: 1   │ {cm[1,0]:8d} │ {cm[1,1]:8d} │")
print(f"  └─────────────┴──────────┴──────────┘")

# Feature importance (for tree models)
if best_model_name != 'Logistic Regression':
    print(f"\n🔍 TOP 10 IMPORTANT FEATURES ({best_model_name}):")
    feature_importance = pd.DataFrame({
        'feature': available_features,
        'importance': best_result['model'].feature_importances_
    }).sort_values('importance', ascending=False)
    
    for idx, row in feature_importance.head(10).iterrows():
        print(f"  {row['feature']:<25} {row['importance']:.4f}")

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETE")
print("=" * 80)

KICKSTARTER DURING-CAMPAIGN MONITOR - MODEL TRAINING

[1/5] Loading data...
Dataset shape: (370451, 27)
Columns: ['ID', 'name', 'blurb', 'category', 'main_category', 'country', 'currency', 'is_us', 'launched', 'deadline', 'launch_month', 'launch_quarter', 'deadline_weekday', 'is_weekend_launch', 'campaign_length', 'is_optimal_duration', 'goal', 'usd_goal_real', 'log_goal', 'pledged', 'usd pledged', 'usd_pledged_real', 'backers', 'state', 'successful', 'category_baseline_success', 'country_baseline_success']

Target variable (success) distribution:
Success column not found

Available features for training: 12/12
Features: ['goal', 'log_goal', 'usd_goal_real', 'launch_month', 'launch_quarter', 'campaign_length', 'is_optimal_duration', 'is_weekend_launch', 'is_us', 'category_baseline_success', 'country_baseline_success', 'deadline_weekday']

[2/5] Preparing data...
  Creating missing features...
Missing values before cleaning: 0
Missing values after cleaning: 0
Final dataset shape: (37045

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("KICKSTARTER DURING-CAMPAIGN MONITOR - MODEL TRAINING")
print("=" * 80)

# -------------------------------------------------------------------
# [1/6] LOAD DATA
# -------------------------------------------------------------------
print("\n[1/6] Loading data...")
df = pd.read_csv('df_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

print(f"\nTarget variable (success) distribution:")
if 'success' in df.columns:
    print(df['success'].value_counts())
else:
    print("Success column not found")

# -------------------------------------------------------------------
# [2/6] DEFINE FEATURES
# -------------------------------------------------------------------
print("\n[2/6] Defining features...")

pre_campaign_features = [
    'goal',
    'log_goal',
    'usd_goal_real',
    'launch_month',
    'launch_quarter',
    'campaign_length',
    'is_optimal_duration',
    'is_weekend_launch',
    'is_us',
    'category_baseline_success',
    'country_baseline_success',
    'deadline_weekday'
]

available_features = [f for f in pre_campaign_features if f in df.columns]
print(f"Available features for training: {len(available_features)}/{len(pre_campaign_features)}")
print(f"Features used: {available_features}")

if len(available_features) == 0:
    print("ERROR: No valid features found in the dataset.")
    exit(1)

# -------------------------------------------------------------------
# [3/6] PREPARE TARGET + CLEAN DATA
# -------------------------------------------------------------------
print("\n[3/6] Preparing data...")

# Target variable
if 'state' in df.columns:
    y = (df['state'] == 'successful').astype(int)
elif 'success' in df.columns:
    y = df['success'].copy()
else:
    print("ERROR: Neither 'state' nor 'success' column found in dataset!")
    exit(1)

X = df[available_features].copy()

print(f"Missing values before cleaning: {X.isnull().sum().sum()}")

# Fill numeric missing values with median
X = X.fillna(X.median(numeric_only=True))

print(f"Missing values after cleaning: {X.isnull().sum().sum()}")

# Remove rows with missing target
valid_idx = ~y.isnull()
X = X[valid_idx]
y = y[valid_idx]

print(f"Final dataset shape: {X.shape}")
print(f"\nSuccess distribution:")
print(y.value_counts())
print(f"Success rate: {y.mean():.2%}")

# -------------------------------------------------------------------
# [4/6] TRAIN / TEST SPLIT
# -------------------------------------------------------------------
print("\n[4/6] Splitting data (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------
# [5/6] TRAIN MODELS
# -------------------------------------------------------------------
print("\n[5/6] Training models...")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'y_pred': y_pred
    }

    print(f"✓ Accuracy:  {accuracy:.2%}")
    print(f"✓ Precision: {precision:.2%}")
    print(f"✓ Recall:    {recall:.2%}")
    print(f"✓ F1-Score:  {f1:.2%}")

# -------------------------------------------------------------------
# [6/6] RESULTS + REPORTS
# -------------------------------------------------------------------
print("\n[6/6] Results Summary")
print("=" * 80)

# Overall comparison table
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1-Score': res['f1']
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Accuracy', ascending=False)

print("\nALL MODELS COMPARISON:")
print(summary_df.round(4).to_string(index=False))

best_model_name = summary_df.iloc[0]['Model']
best_result = results[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"Accuracy: {best_result['accuracy']:.2%}")
print(f"Precision: {best_result['precision']:.2%}")
print(f"Recall: {best_result['recall']:.2%}")
print(f"F1-Score: {best_result['f1']:.2%}")

# -------------------------------------------------------------------
# CLASSIFICATION REPORT FOR EACH MODEL
# -------------------------------------------------------------------
print("\n" + "=" * 80)
print("DETAILED CLASSIFICATION REPORTS (ALL MODELS)")
print("=" * 80)

for name, res in results.items():
    print(f"\n{name}")
    print("-" * 80)

    report_dict = classification_report(
        y_test,
        res['y_pred'],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()

    # Rename rows for nicer output
    report_df.rename(index={
        '0': 'Failed (0)',
        '1': 'Succeeded (1)',
        'accuracy': 'Accuracy',
        'macro avg': 'Macro Avg',
        'weighted avg': 'Weighted Avg'
    }, inplace=True)

    # Keep rows in the order you want
    desired_rows = ['Failed (0)', 'Succeeded (1)', 'Accuracy', 'Macro Avg', 'Weighted Avg']
    report_df = report_df.loc[desired_rows]

    print(report_df.round(3).to_string())

    # Confusion Matrix
    cm = confusion_matrix(y_test, res['y_pred'])
    print(f"\nConfusion Matrix ({name}):")
    print(f"  ┌─────────────┬──────────┬──────────┐")
    print(f"  │             │ Pred: 0  │ Pred: 1  │")
    print(f"  ├─────────────┼──────────┼──────────┤")
    print(f"  │ Actual: 0   │ {cm[0,0]:8d} │ {cm[0,1]:8d} │")
    print(f"  │ Actual: 1   │ {cm[1,0]:8d} │ {cm[1,1]:8d} │")
    print(f"  └─────────────┴──────────┴──────────┘")

# -------------------------------------------------------------------
# TOP FEATURES FOR EACH MODEL
# -------------------------------------------------------------------
print("\n" + "=" * 80)
print("TOP FEATURES FOR EACH MODEL")
print("=" * 80)

for name, res in results.items():
    print(f"\n{name}")
    print("-" * 80)

    model = res['model']

    if name == 'Logistic Regression':
        # Coefficients for scaled features
        importances = np.abs(model.coef_[0])
        feature_df = pd.DataFrame({
            'Feature': available_features,
            'Importance': importances
        }).sort_values('Importance', ascending=False)

    elif hasattr(model, 'feature_importances_'):
        feature_df = pd.DataFrame({
            'Feature': available_features,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False)

    else:
        print("Feature importance not available for this model.")
        continue

    print("Top 10 Features:")
    print(feature_df.head(10).round(4).to_string(index=False))

# -------------------------------------------------------------------
# OPTIONAL: SAVE RESULTS TO CSV FOR SLACK / TEAM SHARING
# -------------------------------------------------------------------
summary_df.to_csv("model_comparison_summary.csv", index=False)
print("\nSaved summary table to: model_comparison_summary.csv")

# Save top features for each model
all_feature_rows = []

for name, res in results.items():
    model = res['model']

    if name == 'Logistic Regression':
        importances = np.abs(model.coef_[0])
    elif hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        continue

    temp_df = pd.DataFrame({
        'Model': name,
        'Feature': available_features,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    all_feature_rows.append(temp_df)

if len(all_feature_rows) > 0:
    all_features_df = pd.concat(all_feature_rows, ignore_index=True)
    all_features_df.to_csv("model_feature_importance.csv", index=False)
    print("Saved feature importance table to: model_feature_importance.csv")

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETE")
print("=" * 80)

KICKSTARTER DURING-CAMPAIGN MONITOR - MODEL TRAINING

[1/6] Loading data...
Dataset shape: (370451, 27)
Columns: ['ID', 'name', 'blurb', 'category', 'main_category', 'country', 'currency', 'is_us', 'launched', 'deadline', 'launch_month', 'launch_quarter', 'deadline_weekday', 'is_weekend_launch', 'campaign_length', 'is_optimal_duration', 'goal', 'usd_goal_real', 'log_goal', 'pledged', 'usd pledged', 'usd_pledged_real', 'backers', 'state', 'successful', 'category_baseline_success', 'country_baseline_success']

Target variable (success) distribution:
Success column not found

[2/6] Defining features...
Available features for training: 12/12
Features used: ['goal', 'log_goal', 'usd_goal_real', 'launch_month', 'launch_quarter', 'campaign_length', 'is_optimal_duration', 'is_weekend_launch', 'is_us', 'category_baseline_success', 'country_baseline_success', 'deadline_weekday']

[3/6] Preparing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Final dataset shape: (3704